# Canadian Retail Sales Forecast: 01 Explore

Source: Statistics Canada, Table **20-10-0056-01**, *Monthly retail trade sales by province and territory* (x 1,000 CAD).

Run each cell, look at the outputs, then answer the **6 decisions** at the bottom. I'll build the forecasting models from your answers.

In [ ]:
import io, zipfile, requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("../data"); DATA.mkdir(exist_ok=True)
URL = "https://www150.statcan.gc.ca/n1/tbl/csv/20100056-eng.zip"
csv_path = DATA / "20100056.csv"

if not csv_path.exists():
    r = requests.get(URL, timeout=120); r.raise_for_status()
    zipfile.ZipFile(io.BytesIO(r.content)).extract("20100056.csv", DATA)

raw = pd.read_csv(csv_path, low_memory=False)
print(raw.shape)
raw.head()

## 1. What's in the table

In [ ]:
print(raw.columns.tolist())
naics_col = [c for c in raw.columns if "NAICS" in c or "Industry" in c][0]
adj_col   = [c for c in raw.columns if "Adjust" in c][0]
for col in ["GEO", adj_col, naics_col]:
    print(f"\n{col} ({raw[col].nunique()} values)")
    print(raw[col].value_counts().head(40).to_string())

In [ ]:
df = raw[["REF_DATE", "GEO", adj_col, naics_col, "VALUE", "STATUS"]].rename(
    columns={adj_col: "adjustment", naics_col: "store_type", "VALUE": "sales_k"})
df["date"] = pd.to_datetime(df["REF_DATE"])
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Months:", df["date"].nunique())
print("\nMissing sales values:", df["sales_k"].isna().sum())
print("\nStatus flags (x = suppressed, E = use with caution, F = too unreliable):")
print(df["STATUS"].value_counts(dropna=False).to_string())

## 2. Canada total: adjusted vs unadjusted

In [ ]:
total = df[(df["GEO"] == "Canada") & (df["store_type"].str.contains("Retail trade", case=False))]
pv = total.pivot_table(index="date", columns="adjustment", values="sales_k") / 1e6  # billions
ax = pv.plot(figsize=(11, 4), title="Canada retail sales ($ billions per month)")
ax.set_ylabel("$B"); plt.show()

Unadjusted shows the real December spikes (useful for planning peaks). Seasonally adjusted removes them (easier to see the trend).

## 3. Provinces (latest 12 months)

In [ ]:
unadj = df[df["adjustment"].str.contains("unadjusted", case=False)]
last12 = unadj[unadj["date"] > unadj["date"].max() - pd.DateOffset(months=12)]
prov = last12[last12["store_type"].str.contains("Retail trade", case=False)]
(prov[prov["GEO"] != "Canada"].groupby("GEO")["sales_k"].sum() / 1e6).sort_values().plot.barh(
    figsize=(8, 5), title="Retail sales, last 12 months ($B)"); plt.show()

## 4. Store types in Canada (latest 12 months)

In [ ]:
types = last12[last12["GEO"] == "Canada"].groupby("store_type")["sales_k"].sum().sort_values() / 1e6
types.plot.barh(figsize=(9, 9), title="Canada sales by store type, last 12 months ($B)"); plt.show()

## 5. Growth and the COVID period

In [ ]:
ca = total[total["adjustment"].str.contains("unadjusted", case=False)].set_index("date")["sales_k"]
yoy = ca.pct_change(12) * 100
ax = yoy.plot(figsize=(11, 4), title="Canada retail sales, year-over-year growth (%)")
ax.axhline(0, color="grey", lw=0.8); ax.axvspan("2020-03", "2021-06", alpha=0.15, label="COVID period")
ax.legend(); plt.show()
print(yoy.tail(12).round(1).to_string())

## Your 6 decisions

Write your answers in the cell below (short is fine).

1. **Data version:** forecast the *unadjusted* numbers (real monthly totals with December spikes) or the *seasonally adjusted* ones (smoother trend)?
2. **What to forecast:** Canada total only, Canada + provinces, or Canada + a few store types? Pick which provinces / store types if so.
3. **Horizon:** how far ahead? (6 months or 12 months)
4. **COVID months (2020–2021):** keep them, drop them, or keep them but flag them in the model?
5. **Backtest:** how many past cutoffs to test on (e.g. forecast each of the last 12 or 24 months as if we didn't know them)?
6. **Business story:** who is the forecast for? (e.g. a payments company planning capacity and merchant support, or a retailer planning stock and staff)

**My answers:**

1. 
2. 
3. 
4. 
5. 
6. 